# 🧹 02 — Preprocessing Teks
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

### Pipeline Preprocessing

```
Teks Mentah
    ↓  1. Case Folding      → semua huruf kecil
    ↓  2. Cleaning          → hapus karakter non-huruf, angka, simbol
    ↓  3. Tokenisasi        → pisah jadi daftar kata
    ↓  4. Stopword Removal  → hapus kata umum (dan, atau, yang, ...)
    ↓  5. Stemming          → potong imbuhan (menggunakan → guna)
Teks Bersih ✅
```

Dijalankan untuk **dua dataset**:
- `skripsi_unila.csv` → kolom `judul` + `topik` → teks bersih untuk query
- `profil_dosen.csv` → kolom `profil_teks` → teks bersih untuk dokumen dosen

> ⚠️ **Pastikan Notebook 01 sudah selesai dijalankan sebelum memulai ini.**

---
## 🔧 LANGKAH 0 — Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))

!pip install -q PySastrawi

import config
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from collections import Counter
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print('✅ Semua library siap.')

---
## 📌 LANGKAH 1 — Load Data Mentah

In [ ]:
# Load data skripsi
df_skripsi = pd.read_csv(config.FILE_SKRIPSI_RAW)

# Load profil dosen
profil_path = os.path.join(config.DATA_RAW, 'profil_dosen.csv')
df_dosen = pd.read_csv(profil_path)

print('✅ Data dimuat.')
print(f'   Skripsi : {len(df_skripsi)} baris | Kolom: {list(df_skripsi.columns)}')
print(f'   Dosen   : {len(df_dosen)} baris  | Kolom: {list(df_dosen.columns)}')
print()
print('Contoh judul skripsi:')
for j in df_skripsi['judul'].head(3):
    print(f'  → {j}')
print()
print('Contoh profil teks dosen (100 karakter):')
for _, row in df_dosen.head(2).iterrows():
    print(f'  [{row["nama_dosen"][:30]}...] {str(row["profil_teks"])[:80]}...')

---
## 📌 LANGKAH 2 — Bangun Pipeline Preprocessing

In [ ]:
# ─── INISIALISASI SASTRAWI ────────────────────────────────────────
stemmer_factory  = StemmerFactory()
stemmer          = stemmer_factory.create_stemmer()

sw_factory       = StopWordRemoverFactory()
stopword_default = set(sw_factory.get_stop_words())

# Stopword tambahan — kata umum di judul skripsi yang tidak informatif
STOPWORD_TAMBAHAN = {
    'berbasis', 'menggunakan', 'dengan', 'pada', 'untuk', 'dalam',
    'studi', 'kasus', 'sistem', 'aplikasi', 'metode', 'analisis',
    'implementasi', 'rancang', 'bangun', 'pembangunan', 'pengembangan',
    'perancangan', 'penerapan', 'pembuatan', 'pendekatan', 'terhadap',
    'berdasarkan', 'studi kasus', 'dan', 'atau', 'yang', 'di', 'ke',
    'dari', 'adalah', 'sebagai', 'secara', 'study', 'case',
}

ALL_STOPWORDS = stopword_default | STOPWORD_TAMBAHAN

print(f'✅ Stemmer & Stopword siap.')
print(f'   Stopword default Sastrawi  : {len(stopword_default)} kata')
print(f'   Stopword tambahan          : {len(STOPWORD_TAMBAHAN)} kata')
print(f'   Total stopword             : {len(ALL_STOPWORDS)} kata')

In [ ]:
# ─── FUNGSI PREPROCESSING ─────────────────────────────────────────

def clean_text(text):
    """Step 1 & 2: Case folding + hapus karakter tidak relevan."""
    if pd.isna(text) or text == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)   # hapus non-huruf (angka, simbol, tanda baca)
    text = re.sub(r'\s+', ' ', text).strip() # rapikan spasi
    return text


def tokenize(text):
    """Step 3: Tokenisasi sederhana (split by spasi)."""
    return text.split()


def remove_stopwords(tokens):
    """Step 4: Hapus stopword dari list token."""
    return [t for t in tokens if t not in ALL_STOPWORDS and len(t) > 2]


def stem_tokens(tokens):
    """Step 5: Stemming setiap token menggunakan Sastrawi."""
    return [stemmer.stem(t) for t in tokens]


def preprocess(text, apply_stemming=True):
    """
    Pipeline lengkap: clean → tokenize → stopword → stem → gabung.
    Parameter apply_stemming=False digunakan untuk model BERT
    (BERT lebih baik menerima teks asli, bukan hasil stemming).
    """
    cleaned = clean_text(text)
    tokens  = tokenize(cleaned)
    tokens  = remove_stopwords(tokens)
    if apply_stemming:
        tokens = stem_tokens(tokens)
    return ' '.join(tokens)


def preprocess_for_bert(text):
    """Preprocessing ringan untuk BERT — tanpa stemming, hanya clean & stopword."""
    return preprocess(text, apply_stemming=False)


print('✅ Fungsi preprocessing siap.')

In [ ]:
# ─── UJI FUNGSI PREPROCESSING ─────────────────────────────────────
contoh_judul = [
    'Implementasi Deep Learning untuk Deteksi Penyakit Tanaman Berbasis Citra Digital',
    'Pembangunan Sistem Informasi Manajemen Arsip Berbasis Web Menggunakan Laravel',
    'Penerapan Metode K-Nearest Neighbor untuk Klasifikasi Data Mahasiswa',
]

print('🔬 UJI PREPROCESSING:\n')
print(f'{"INPUT":<65} {"→ TF-IDF (stem)":<40} {"→ BERT (no stem)"}')
print('-' * 140)
for judul in contoh_judul:
    hasil_stem  = preprocess(judul)
    hasil_bert  = preprocess_for_bert(judul)
    print(f'{judul[:63]:<65} {hasil_stem[:38]:<40} {hasil_bert[:40]}')

# Tampilkan detail step-by-step untuk satu contoh
print('\n' + '='*60)
print('📋 DETAIL STEP-BY-STEP untuk contoh pertama:')
print('='*60)
teks = contoh_judul[0]
print(f'  Input     : {teks}')
print(f'  Clean     : {clean_text(teks)}')
tokens = tokenize(clean_text(teks))
print(f'  Tokenize  : {tokens}')
tokens_sw = remove_stopwords(tokens)
print(f'  Stopword  : {tokens_sw}')
tokens_st = stem_tokens(tokens_sw)
print(f'  Stem      : {tokens_st}')
print(f'  Output    : {", ".join(tokens_st)}')

---
## 📌 LANGKAH 3 — Preprocessing Data Skripsi

In [ ]:
# ─── PREPROCESSING JUDUL SKRIPSI ─────────────────────────────────
print('⏳ Memproses teks skripsi...')

# Teks input = judul + topik (sudah digabung di notebook 01)
df_skripsi['teks_bersih']      = df_skripsi['teks_input'].apply(preprocess)
df_skripsi['teks_bersih_bert'] = df_skripsi['teks_input'].apply(preprocess_for_bert)

print(f'✅ Selesai. {len(df_skripsi)} baris diproses.')
print()
print('Contoh hasil preprocessing:')
for i in range(3):
    row = df_skripsi.iloc[i]
    print(f'  [{i+1}] Judul    : {row["judul"][:70]}')
    print(f'       TF-IDF  : {row["teks_bersih"][:70]}')
    print(f'       BERT    : {row["teks_bersih_bert"][:70]}')
    print()

---
## 📌 LANGKAH 4 — Preprocessing Profil Dosen

In [ ]:
# ─── PREPROCESSING PROFIL TEKS DOSEN ─────────────────────────────
print('⏳ Memproses teks profil dosen...')

df_dosen['profil_bersih']      = df_dosen['profil_teks'].apply(preprocess)
df_dosen['profil_bersih_bert'] = df_dosen['profil_teks'].apply(preprocess_for_bert)

print(f'✅ Selesai. {len(df_dosen)} dosen diproses.')
print()
print('Contoh profil dosen (80 karakter):')
for _, row in df_dosen.iterrows():
    nama   = row['nama_dosen'][:35]
    profil = str(row['profil_bersih'])[:80]
    print(f'  [{nama:<35}] {profil}...')

---
## 📌 LANGKAH 5 — Analisis Kualitas Teks

In [ ]:
# ─── STATISTIK TOKEN ─────────────────────────────────────────────
df_skripsi['n_token_raw']    = df_skripsi['teks_input'].apply(lambda x: len(str(x).split()))
df_skripsi['n_token_bersih'] = df_skripsi['teks_bersih'].apply(lambda x: len(str(x).split()))
df_skripsi['reduksi_pct']    = ((df_skripsi['n_token_raw'] - df_skripsi['n_token_bersih']) / df_skripsi['n_token_raw'] * 100).round(1)

print('📊 STATISTIK TOKEN DATA SKRIPSI:')
print(f'  Rata-rata token sebelum preprocessing : {df_skripsi["n_token_raw"].mean():.1f}')
print(f'  Rata-rata token sesudah preprocessing : {df_skripsi["n_token_bersih"].mean():.1f}')
print(f'  Rata-rata reduksi                     : {df_skripsi["reduksi_pct"].mean():.1f}%')
print(f'  Token maks (raw)                      : {df_skripsi["n_token_raw"].max()}')
print(f'  Token maks (bersih)                   : {df_skripsi["n_token_bersih"].max()}')
print()

df_dosen['n_token_profil'] = df_dosen['profil_bersih'].apply(lambda x: len(str(x).split()))
print('📊 STATISTIK TOKEN PROFIL DOSEN:')
print(f'  Min token  : {df_dosen["n_token_profil"].min()}')
print(f'  Max token  : {df_dosen["n_token_profil"].max()}')
print(f'  Rata-rata  : {df_dosen["n_token_profil"].mean():.1f}')
print()
print(df_dosen[['nama_dosen','n_token_profil']].sort_values('n_token_profil', ascending=False).to_string(index=False))

In [ ]:
# ─── TOP KATA PALING SERING ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Kata terbanyak di judul skripsi
all_tokens_skripsi = ' '.join(df_skripsi['teks_bersih'].dropna()).split()
top_skripsi = Counter(all_tokens_skripsi).most_common(20)
words_s, counts_s = zip(*top_skripsi)
axes[0].barh(list(reversed(words_s)), list(reversed(counts_s)), color='steelblue', edgecolor='white')
axes[0].set_title('Top 20 Kata — Judul Skripsi (setelah preprocessing)', fontweight='bold')
axes[0].set_xlabel('Frekuensi')

# Kata terbanyak di profil dosen
all_tokens_dosen = ' '.join(df_dosen['profil_bersih'].dropna().astype(str)).split()
top_dosen = Counter(all_tokens_dosen).most_common(20)
words_d, counts_d = zip(*top_dosen)
axes[1].barh(list(reversed(words_d)), list(reversed(counts_d)), color='darkorange', edgecolor='white')
axes[1].set_title('Top 20 Kata — Profil Dosen (setelah preprocessing)', fontweight='bold')
axes[1].set_xlabel('Frekuensi')

plt.tight_layout()
plot_path = os.path.join(config.RESULTS_DIR, 'top_kata.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Grafik tersimpan: {plot_path}')

In [ ]:
# ─── DISTRIBUSI PANJANG TOKEN ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_skripsi['n_token_raw'], bins=20, alpha=0.6, color='steelblue', label='Sebelum')
axes[0].hist(df_skripsi['n_token_bersih'], bins=20, alpha=0.6, color='darkorange', label='Sesudah')
axes[0].set_title('Distribusi Jumlah Token — Judul Skripsi', fontweight='bold')
axes[0].set_xlabel('Jumlah Token')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

axes[1].bar(df_dosen['nama_dosen'].apply(lambda x: x.split(',')[0]),
            df_dosen['n_token_profil'], color='teal', edgecolor='white')
axes[1].set_title('Panjang Profil Teks per Dosen (token)', fontweight='bold')
axes[1].set_xlabel('Dosen')
axes[1].set_ylabel('Jumlah Token')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'distribusi_token.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── CEK DATA BERMASALAH ─────────────────────────────────────────
print('🔍 CEK DATA BERMASALAH:\n')

# Skripsi dengan hasil preprocessing sangat pendek (< 3 token)
pendek = df_skripsi[df_skripsi['n_token_bersih'] < 3]
print(f'  ⚠️  Skripsi dengan teks bersih < 3 token: {len(pendek)}')
if len(pendek) > 0:
    print(pendek[['judul','teks_bersih']].to_string())

# Dosen dengan profil sangat pendek (< 10 token)
dosen_pendek = df_dosen[df_dosen['n_token_profil'] < 10]
print(f'\n  ⚠️  Dosen dengan profil < 10 token: {len(dosen_pendek)}')
if len(dosen_pendek) > 0:
    print(dosen_pendek[['nama_dosen','n_token_profil']].to_string())
    print('  → Pertimbangkan menambah data publikasi manual untuk dosen ini.')

if len(pendek) == 0 and len(dosen_pendek) == 0:
    print('  ✅ Semua data memiliki token yang cukup!')

---
## 📌 LANGKAH 6 — Simpan Data Bersih

In [ ]:
# ─── SIMPAN DATA SKRIPSI BERSIH ───────────────────────────────────
df_skripsi.to_csv(config.FILE_SKRIPSI_CLEAN, index=False, encoding='utf-8-sig')
print(f'💾 skripsi_clean.csv  → {config.FILE_SKRIPSI_CLEAN}')
print(f'   Baris   : {len(df_skripsi)}')
print(f'   Kolom   : {list(df_skripsi.columns)}')
print()

# ─── SIMPAN PROFIL DOSEN BERSIH ──────────────────────────────────
df_dosen.to_csv(config.FILE_DOSEN_CLEAN, index=False, encoding='utf-8-sig')
print(f'💾 profil_dosen_clean.csv → {config.FILE_DOSEN_CLEAN}')
print(f'   Baris   : {len(df_dosen)}')
print(f'   Kolom   : {list(df_dosen.columns)}')

In [ ]:
# ─── PREVIEW AKHIR ────────────────────────────────────────────────
print('📋 PREVIEW DATA SKRIPSI BERSIH (5 baris):')
display_cols = ['judul','teks_bersih','teks_bersih_bert','pembimbing']
df_skripsi[display_cols].head(5).style.set_properties(**{'text-align':'left'})

In [ ]:
print('📋 PREVIEW PROFIL DOSEN BERSIH:')
display_cols_d = ['nama_dosen','bidang','n_token_profil','profil_bersih']
# Potong profil_bersih agar tidak terlalu panjang
df_show = df_dosen[display_cols_d].copy()
df_show['profil_bersih'] = df_show['profil_bersih'].astype(str).str[:100] + '...'
df_show

---
## ✅ Selesai — Ringkasan Notebook 02

| File Output | Kolom Penting | Keterangan |
|-------------|---------------|------------|
| `data/processed/skripsi_clean.csv` | `teks_bersih` | Untuk TF-IDF |
| `data/processed/skripsi_clean.csv` | `teks_bersih_bert` | Untuk IndoBERT/SBERT |
| `data/processed/profil_dosen_clean.csv` | `profil_bersih` | Profil dosen untuk TF-IDF |
| `data/processed/profil_dosen_clean.csv` | `profil_bersih_bert` | Profil dosen untuk BERT |
| `results/top_kata.png` | — | Visualisasi kata terbanyak |
| `results/distribusi_token.png` | — | Distribusi panjang teks |

### 🗺️ Langkah Berikutnya:
> **`03_tfidf_baseline.ipynb`** — TF-IDF vectorizer + Cosine Similarity + evaluasi Top-K